# TP 2 : Analyse d'un LLM et benchmarking

## 1. Analyse d'un LLM en local

### 1.1 Installation

#### 1.1.1 Configuration de VS Code

Voir `tp2.pptx` sur Moodle.

#### 1.1.2 Installation des librairies

In [1]:
# Installation des bibliothèques  (Ne faire qu'une fois)
%pip install transformers accelerate huggingface_hub
%pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 24.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 29.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 49.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 71.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 62.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 36.9 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 23.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 9.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33/33 [transformers] [transformers]ub]
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 15.8 

#### 1.1.3 Vérification de l'environnement



In [2]:
# Vérification de l'environnement
import torch
import transformers

print(f"PyTorch version   : {torch.__version__}")
print(f"Transformers ver  : {transformers.__version__}")
print(f"CUDA disponible   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM              : {total_mem:.1f} GB")
else:
    print("⚠️  Pas de GPU détecté — le CPU sera utilisé (plus lent)")

PyTorch version   : 2.10.0
Transformers ver  : 5.3.0
CUDA disponible   : False
⚠️  Pas de GPU détecté — le CPU sera utilisé (plus lent)


### 1.2 Qwen

#### 1.2.1 Qwen/Qwen3-0.6B-Base

Nous étudions d'abord un modèle LLM brut, dont l'identifiant modèle est : `Qwen/Qwen3-0.6B-Base`

1. Expliquer les différents parties de l'identifiant. Pourquoi utilisons-nous la version `0.6B` dans ce TP ?
#qwen c'est l'organisation
#le 3 c'est la version du modèle 
#0.6B la taille du modèle, le nombre de paramètre
#la variante
#on utilise celle-ci pour que on puisse le lancer sur l'ordinateur
2. Tester le code ci-dessous. Que fait-il ?
#il demande au LLM de répondre à la question : Qu'est ce que l'inteligence artificielle ? 
#ça réponse : Qu'est-ce que l'intelligence artificielle ?' est un outil de calcul? L'IA peut-elle être utilisée pour créer de l'art? Quels sont les principaux défis à relever pour l'IA dans le domaine de l'art?

L'intelligence artificielle (IA) est un domaine théorique et pratique qui étudie comment les ordinateurs peuvent raisonner, écrire, et interagir avec des personnes humaines. Elle utilise des algorithmes et des techniques pour apprendre, raisonner, et résoudre des problèmes complexes. L'intelligence artific
#c'est une réponse qui commence avec des question et incomplète
3. Commenter chaque ligne du code pour comprendre son rôle.
4. Tester différents prompts.
5. Changer le paramètre `temperature`. Qu'est-ce que vous observez ? Que fait concrètement ce paramètre ?
#plus la temperature est haut plus les réponses sont différente pour les même questions, c'est un système d'aléatoire.
6. Et si vous remplacez l'argument `temperature` par `do_sample=False` et que vous exécutez le bloc le deux fois de suite ?
#c'est l'équivalent de temperature=0 ce qui veut dire que toutes les réponses seront les même

In [2]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device="mps",
    torch_dtype=torch.float32
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [9]:
prompt = "Qu'est ce que la philsosophie ?'"  # chaîne d'entrée envoyée au modèle (prompt). Remarquer l'apostrophe finale possiblement accidentelle.
# prompt = "Ainsi donc"                                    # alternative de prompt commentée (on peut décommenter pour tester)
 
result = gen(                                            # appel à la pipeline 'gen' (text-generation) pour produire du texte
    prompt,                                              # prompt passé en entrée
    do_sample=False,                                     # contrôle la "créativité" / diversité (plus élevé => sorties plus variées), l'aléatoire
    max_new_tokens=120,                                  # nombre maximal de tokens que le modèle peut générer
    return_full_text=True,                               # si True, la sortie contient le prompt + le texte généré ; si False, seulement le texte généré
)
 
print(                                                   # affiche la réponse dans la sortie standard
    "\nRÉPONSE\n\n" +                                     # préfixe d'affichage pour lisibilité
    result[0]["generated_text"]                          # extrait le texte généré du premier (et typiquement unique) résultat retourné par la pipeline
)


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Qu'est ce que la philsosophie ?'s a été développée par des philosophes comme Socrate, Platon, et Aristote. Elle est une méthode de pensée qui vise à comprendre les vérités fondamentales de la nature humaine et de l'univers. La philsophie est souvent considérée comme une science qui explore les questions fondamentales de la vie, tels que la liberté, la justice, la valeur de l'homme, et la nature de l'existence.

Qu'est-ce que la philosophie de la liberté?'s une approche philosophique qui examine


#### 1.2.2 Qwen/Qwen3-0.6B

Nous utilisons maintenant la version sans `Base` de `Qwen3-0.6B-Base`.

1. Que signifie ce changement dans l'identifiant ?
#sans base fait passé ssur la version brute, le modèle s'attend à des prompt plus clair et moin extravagent
2. Tester le code ci-dessous. Qu'est-ce que est différent ?
#les réponses sont plus serieuses 
3. Commenter les nouvelles lignes pour expliquer leurs rôles ?
3. Tester différents `content` pour `user`.
4. Décommenter les lignes `system`. Tester différents `content`. 
#pour changer ça magnière de répondre
5. Analysez le contenu du prompt. Quels sont ses différents éléments et que veulent-ils dire ? 
#il a pusieurs étapes et façon de répondre
6. Et que se passe-t-il si vous remplacez le `prompt` par une question ou un texte quelconque ?
#
7. Que se passe-t-il avec `enable_thinking=True` ?
#il réflechit beaucoup plus mais répond en anglais

In [10]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device="mps",
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [28]:
# ---- Début des nouvelles lignes
messages = [
    # {"role": "system", "content": "You only answer with one word."},
    {"role": "system", "content": "tu réponds comme si tu étais un chef cuisinier expérimenté qui donne des conseils de recette détaillés et précis."},
    {"role": "user",   "content": "Donne moi la recette idéal pour un gâteau au chocolat?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages,                            # la liste des messages (system/user) à incorporer dans le prompt
    tokenize=False,                     # ne pas tokeniser maintenant (la pipeline s'en chargera si besoin)
    add_generation_prompt=True,          # ajouter le suffixe de génération (ex: invite pour l'assistant)
    enable_thinking=True              # option spécifique: active/désactive un bloc "thinking" explicatif si supporté
)

# prompt = "Ainsi donc"
# ---- Fin des nouvelles lignes

result = gen(
    prompt,
    do_sample=False,                     # rendu déterministe : pas d'échantillonnage aléatoire
    max_new_tokens=700,                 # limite du nombre de tokens à générer
    return_full_text=False,            # si False, renvoie uniquement le texte généré (pas le prompt)
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]      # extraction et affichage du texte généré par la pipeline
)

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

<think>
Okay, the user is asking for the ideal chocolate cake recipe. Let me start by recalling the key elements of a classic chocolate cake. First, the ingredients are important. I need to mention the main components: flour, sugar, cocoa, butter, eggs, and maybe some milk. It's important to note that the ratio of flour to sugar is crucial for texture. Maybe suggest using a 1:1.5 ratio for a cake that's not too dense.

Next, the preparation steps. I should outline the steps clearly, maybe starting with preheating the oven. Then, mixing the ingredients. It's a common mistake to overmix, so I should remind the user to do it just enough to combine the ingredients without making it too runny. Also, the baking time is a key factor. I need to mention the standard baking time and temperature, which is around 350°F (177°C).

I should also consider variations. The user might be looking for something unique, so suggesting alternatives like using a cake mix or adding a bit of vanilla co

### 1.3 TinyLlama

Voici un autre modèle : `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

1. Tester quelques questions et comparer les réponses à celles de Qwen
2. Tester différents paramètres.
2. Que pensez-vous de la capacité de TinyLlama à suivre les instructions `system`?
#c'est un peu mieux et plus inteligent mais se répéte vite

In [29]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device=-1,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [30]:
messages = [
    # {"role": "system", "content": "You only answer with one word."},
    {"role": "system", "content": "You answer with a detailed and precise recipe."},
    # {"role": "system", "content": "You are a pessimistic and poetic sci-fi writer that warns the user of the dangers of technology."},
    {"role": "user",   "content": "Donne moi la recette idéal pour un gâteau au chocolat?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
)

# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=0.1,
    max_new_tokens=700,
    return_full_text=False, # ne retourne pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=700) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Certainement! 

Si vous souhaitez un gâteau au chocolat idéal, vous pouvez essayer la recette suivante:

Ingredients:
- 240g de chocolat haché
- 120g de sucre en polenta
- 120g de farine
- 120g de levure chimique
- 120g de sucre à café
- 120g de cacao liquide
- 120g de cacao poudre
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir poudre
- 120g de cacao noir liquide
- 120g de cacao noir po

### 1.4 SmolLM

1. Tester quelques questions et comparer les réponses à celles de Qwen et TinyLlama.
2. Tester différents paramètres.
2. Que pensez-vous de la capacité de SmolLM à suivre les instructions `system`?
#bonne réponse est plus détaillée. Mais toujours pas bonne et pas logique

In [31]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device=-1,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [34]:
messages = [
    # {"role": "system", "content": "You only answer with one word."},
    {"role": "system", "content": "You answer with a detailed and precise recipe."},
    # {"role": "system", "content": "You are a pessimistic and poetic sci-fi writer who warns the user of the dangers of technology."},
    {"role": "user",   "content": "Donne moi la recette idéal pour un gâteau au chocolat?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=0.7,
    max_new_tokens=300,
    return_full_text=False, # ne pas retourner pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Voici la recette idéale pour un gâteau au chocolat:

Ingredients:
- 200 g de sucre brun roux
- 200 g de sucre blanc
- 3 g de levure
- 2 g de levure levée
- 100 g de beurre mielier
- 5 g de coca
- 125 g d'œufs
- 100 g de beurre de graine
- 100 g de chocolat fondu
- 150 g de fraises

Instructions:

1. Dans un frigo, remuer les 200 g de sucre roux, le sucre blanc et les 2 g de levures jusqu'à ce qu'ils soient bien moussé.

2. Ensuite, ajoutez le levure levée et la levure, le beurre mielier et la coca, et remuer jusqu'à ce qu'elles soient moussées.

3. Ensuite, ajoutez les œufs, le beurre de graine et le chocolat fondu. Remuer jusqu'à ce qu'ils soient moussés.

4. En


### 1.5 Pour aller plus loin

* Tester `HuggingFaceTB/SmolLM3-3B` (plus lent)
* Explorer d'autres modèles sur [HuggingFace](https://huggingface.co) 


## 2 Benchmarking

### 2.1 MMLU

#### 2.1.1 Télécharger et explorer les données du benchmark

1. Exécuter le code pour télécharger la banque de question du MMLU sur les mathématiques élémentaires.

In [33]:
%pip install datasets # ne faire qu'une fois

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 42.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 25.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [datasets]/17 [datasets]]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datasets import load_dataset
ds_elementary_mathematics = load_dataset("cais/mmlu", "elementary_mathematics")

2. Analyser la structure du jeu de données
3. Explorer quelques questions du benchmark

In [ ]:
ds_elementary_mathematics
# ds_elementary_mathematics['dev'][0]

4. Explorer la liste des sujets abordés par le MMLU
5. Explorer les questions de quelques autres sujets 

In [ ]:
from datasets import get_dataset_config_names

subjects = get_dataset_config_names("cais/mmlu")
print(f"{len(subjects)} matières disponibles :\n")
for s in sorted(subjects):
    print(f"  {s}")

#### 2.1.2 Tester avec des modèles locaux

1. Tester quelques questions avec différents modèles locaux

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device=-1,
    torch_dtype=torch.float32
)

In [ ]:
question = ds_elementary_mathematics['dev'][0]['question'] + "\n" + \
    " | ".join(f"{chr(65+i)}. {c}" for i, c in enumerate(ds_elementary_mathematics['dev'][0]['choices'])) + \
    "\n Only reply with a letter."


messages = [
    {"role": "system", "content": "Answer with a single letter only (A, B, C or D):"},
    {"role": "user",   "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=True
)

result = gen(
    prompt,    
    do_sample=False, # Modèle déterministique
    max_new_tokens=1000,
    return_full_text=False,
)

print(
    result[0]["generated_text"]
)

2. Testez le modèle plus systématiquement sur plusieurs questions, en vous aidant du code ci-dessous.
3. Testez avec différents benchmarks et paramètre de modèles et notez vos impressions.
4. Pourquoi est-ce que `do_sample=False` ?

In [ ]:
import json, re

results = []

for item in ds_elementary_mathematics['dev']:
    question_text = item['question'] + "\n" + \
        " | ".join(f"{chr(65+i)}. {c}" for i, c in enumerate(item['choices'])) + \
            "\n\nReply with a single letter: A, B, C or D. Nothing else."
    messages = [
        {"role": "system", "content": "Reply with a single letter: A, B, C or D. Nothing else."},
        {"role": "user", "content": question_text}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                            add_generation_prompt=True,
                                            enable_thinking=False)
    
    result = gen(prompt, do_sample=False, max_new_tokens=10,
                 return_full_text=False)
    
    response = result[0]["generated_text"].strip()
    match = re.search(r'\b([A-D])\b', response)
    answer = ord(match.group(1)) - ord('A') if match else -1
    results.append(answer)

print(results)

# Comparer avec les vraies réponses
correct = [item['answer'] for item in ds_elementary_mathematics['dev']]
accuracy = sum(p == c for p, c in zip(results, correct)) / len(correct)
print(f"Accuracy : {accuracy:.1%}")

#### 2.2 Tester avec les modèles majeurs

1. Essayer de tester des modèles majeurs sur le benchmark et de calculer le taux de succès.
2. Comparer vos scores aux résultats sur les leaderboards de quelques sites de référence.
3. Nommer quelques limitations de votre test.

### 2.3 Autres benchmarks (Lecture 3, pour le mardi 10 Mars)

Il existent de nombreux autres benchmarks. 

1. Faites une recherche pour en trouver 2 ou 3, les plus variés possibles. Vous pouvez essayer de trouver des benchmarks qui testent des sujets qui vous intéressent (science, programmation, mathématiques, écriture de fiction, etc.).
2. Créer un petit tableau avec des informations générales sur les benchmarks.
3. Quelles compétences des IA mesurent vos benchmarks ?
4. Trouver quelques questions et regardez les réponses des modèles locaux, ainsi que ceux des grands modèles. Quelles sont vos impressions ?
5. Cherchez les résultats des modèles sur vos benchmarks sur des sites de références. Comment ont évolué les scores des meilleurs modèles depuis la publication du benchmark.